# CNN Training Notebook

The CNN's job in this game is to classify humanoid images into four categories: **healthy**, **injured**, **zombie**, and **corpse**. When the game runs in CNN-assist mode, each humanoid's image is passed through this network and the predicted class is shown to the player (or used directly by the RL agent). A well-trained CNN makes the game meaningfully harder for the RL agent — it has to act on imperfect information rather than ground truth.

This notebook trains `DefaultCNN` a lightweight convolutional classifier on the consolidated humanoid image dataset, then saves the weights to `models/weights/baselineCNN.pth`. It mirrors `training/train_cnn.py` exactly, with argparse replaced by hardcoded constants you can edit directly.

In [1]:
%matplotlib inline

In [2]:
import os
import sys
from pathlib import Path

# Ensure the project root is the working directory and is on sys.path,
# so local packages (game_interfaces, gameplay, models) are importable
# and relative paths like DATA_DIR="data" resolve correctly.
# Works whether Jupyter was launched from bwsi-sgai/ or training/.
_here = Path().resolve()
_root = _here.parent if _here.name == "training" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import os
import random
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

from game_interfaces.data_parser import datarow_to_state
from gameplay.constants import CLASSIFIER_MODEL_PATH, DATA_DIR
from gameplay.enums import State
from gameplay.humanoid import MAP_CLASS_STR_TO_INT
from models.DefaultCNN import DefaultCNN

## Hyperparameters

These are the same constants defined at the top of `train_cnn.py`. Edit them here to run experiments.

Key things to try:
- **`NUM_EPOCHS`** — more epochs = more passes through the data. Watch for overfitting: if val accuracy starts dropping while train accuracy keeps rising, you've trained too long.
- **`BATCH_SIZE`** — number of images processed per gradient step. Larger batches are more stable but use more memory. With a small dataset, smaller batches (4–8) often work better.
- **`LEARNING_RATE`** — how large each gradient step is. If loss diverges (shoots up), lower this. If training is very slow, try increasing it.
- **`VAL_SPLIT`** — fraction of data reserved for validation (not used during training). 0.2 means 20% validation, 80% training.

### Accuracy improvements to try

With a small dataset like this one, the model tends to overfit quickly (train accuracy → 100%, val accuracy plateaus or drops). Three approaches that help:

- **Data augmentation**: two transforms are commented out in the data loading cell below — `ColorJitter` and `RandomRotation`. Try enabling them one at a time and watch the training curves. Each one changes what the model sees in a different way. What do you predict will happen?
- **Regularization**: pass `weight_decay=1e-4` to `torch.optim.Adam`, or add `Dropout` layers to `DefaultCNN` — penalizes large weights and reduces overfitting.
- **Early stopping**: track the best val accuracy across epochs and save weights at the peak instead of the final epoch — the baseline run peaks around epoch 7–8 before overfitting sets in.

In [4]:
# ── Hyperparameters — edit these to experiment ──────────────────────────────
NUM_EPOCHS: int = 10
BATCH_SIZE: int = 8
LEARNING_RATE: float = 0.001
VAL_SPLIT: float = 0.2         # fraction of data held out for validation

SEED: int | None = 42

NUM_CLASSES: int = len(State)  # 4 classes: healthy, injured, zombie, corpse

# Where to load data from, where to save weights, and where to write plots
TRAINING_DATA_DIR: str = DATA_DIR
SAVE_PATH: str = CLASSIFIER_MODEL_PATH
CNN_LOG_DIR: str = "training/CNN-logs"

print(f"Number of classes: {NUM_CLASSES}")
print(f"Weights will be saved to: {SAVE_PATH}")
print(f"Plots will be saved to: {CNN_LOG_DIR}")

Number of classes: 4
Weights will be saved to: models\weights\baselineCNN.pth
Plots will be saved to: training/CNN-logs


## Seeding

A fixed seed ensures your train/val split and training trajectory are the same every run. This is essential when comparing two experiments — without seeding, you can't tell whether a result difference came from your change or from random variation in the data split or weight initialization.

We seed Python's `random`, NumPy, and PyTorch (CPU and GPU) so every random operation is deterministic.

In [5]:
def set_seed(seed: int) -> None:
    """Seed all random number generators for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


if SEED is not None:
    set_seed(SEED)
    print(f"Seeded with {SEED}.")
else:
    print("Seeding disabled — results will vary between runs.")

Seeded with 42.


## Dataset

### Label mapping

The metadata CSV has columns like `Class` ("Default" or "Zombie") and `Injured` (True/False). We need to convert those two columns into a single integer class label that PyTorch can train on. The mapping is:

| Class | Injured | → | Label |
|-------|---------|---|-------|
| Default | False | → | HEALTHY |
| Default | True  | → | INJURED |
| Zombie  | False | → | ZOMBIE  |
| Zombie  | True  | → | CORPSE  |

This matches `datarow_to_state()` in the game's data parser, so the classes the CNN learns during training are exactly the classes the game uses during inference.

### HumanoidDataset

`HumanoidDataset` is a PyTorch `Dataset` — a class that tells PyTorch how to load one (image, label) pair given an index. DataLoaders use this to serve batches during training. It reads the metadata CSV once at construction and builds a list of image paths and labels; images are loaded from disk on demand in `__getitem__`.

In [6]:
def _datarow_to_label(row: pd.Series) -> int:
    """Map a metadata CSV row to an integer class label.

    Delegates to datarow_to_state() so the CNN trains on exactly the same
    class definitions the game uses at inference time. Customize state
    mapping there, not here.
    """
    return MAP_CLASS_STR_TO_INT[datarow_to_state(row)]


class HumanoidDataset(Dataset):
    """PyTorch Dataset wrapping the consolidated humanoid image dataset."""

    img_data_root: str
    transform: Any
    image_paths: list[str]
    labels: list[int]

    def __init__(
        self,
        metadata_path: str,
        img_data_root: str,
        transform: Any = None,
    ) -> None:
        self.img_data_root = img_data_root
        self.transform = transform
        df = pd.read_csv(metadata_path)
        self.image_paths = df["Filename"].tolist()
        self.labels = [_datarow_to_label(row) for _, row in df.iterrows()]

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> tuple[Any, int]:
        img_path = os.path.join(self.img_data_root, self.image_paths[idx])
        with Image.open(img_path) as image_:
            image_ = image_.convert("RGB")
            if self.transform:
                image_ = self.transform(image_)
        return image_, self.labels[idx]

## Data Loading

We apply three transforms to every image before feeding it to the network:

1. **Resize to 512×512** — all images become the same size so batching works.
2. **ToTensor** — converts a PIL image (pixel values 0–255) to a float tensor (values 0.0–1.0).
3. **Normalize** — subtracts the ImageNet mean and divides by the ImageNet standard deviation, channel by channel. We use ImageNet statistics even though these aren't ImageNet images because `DefaultCNN`'s convolutional filters were originally designed for ImageNet-normalized input. Using the same normalization at training time and inference time is what matters — the game's `ImageClassifier` applies the same transform when calling the model.

The dataset is split 80/20 into train and validation sets. The split uses a fixed generator seed so it's the same every run.

In [7]:
metadata_path = os.path.join(TRAINING_DATA_DIR, "consolidated_metadata.csv")

device = (
    torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
)
print(f"Device: {device}")

# Normalization statistics must match CNNClassifier.classify() exactly —
# the same mean/std applied at training time must be applied at inference
# time, or the model sees a shifted input distribution and performs poorly.
_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],  # ImageNet channel statistics
)

# Training transform: augmentation + normalize.
# Augmentation shows the model slightly different versions of each image
# every epoch, which helps it learn generalizable features instead of
# memorizing exact pixels. This reduces overfitting on small datasets —
# but there's a hard limit: no matter how many ways you transform 160
# images, you can't manufacture information that isn't there. Augmentation
# buys you something, but it's not a substitute for more training data.
#
# RandomHorizontalFlip is the one augmentation active in this baseline.
# Mirroring left↔right preserves both class-distinguishing features in
# this dataset (color and upright/prone orientation), so it's always safe.
# It's also the only one that consistently improves validation accuracy
# without slowing or breaking convergence on a dataset this small.
#
# The commented-out transforms below are experiments for you to try.
# Each one changes what the model sees during training in a different way —
# run them and watch what happens to the training curves.
#
# Augmentation is applied ONLY to training images — validation images stay
# clean so accuracy numbers reflect real generalisation, not augmented luck.
train_transforms = transforms.Compose(
    [
        transforms.Resize((512, 512)),
        transforms.RandomHorizontalFlip(),
        # transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
        # ^ ColorJitter randomly shifts brightness, contrast, saturation, and hue.
        # Think about which visual feature most separates zombies from humans
        # in this dataset, then predict what ColorJitter does to that signal.
        # transforms.RandomRotation(degrees=15),
        # ^ RandomRotation tilts images up to ±15°. Two of the four classes
        # differ by orientation (upright vs. prone). What happens to that
        # signal when a lying-down image gets rotated 90°?
        transforms.ToTensor(),
        _normalize,
    ]
)

# Validation transform: resize + normalize only — no augmentation.
val_transforms = transforms.Compose(
    [
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        _normalize,
    ]
)

# Build two dataset instances over the same files — one per transform.
# We then assign the same split indices to both so train and val images
# never overlap, while each split gets its own transform pipeline.
train_dataset = HumanoidDataset(metadata_path, TRAINING_DATA_DIR, transform=train_transforms)
val_dataset = HumanoidDataset(metadata_path, TRAINING_DATA_DIR, transform=val_transforms)

n_total = len(train_dataset)
val_size = max(1, int(n_total * VAL_SPLIT))
train_size = n_total - val_size
if train_size < 1:
    raise ValueError(
        f"Not enough data for training — dataset has {n_total} sample(s) but "
        f"VAL_SPLIT={VAL_SPLIT} reserves {val_size} for validation, leaving 0 for training. "
        "Reduce VAL_SPLIT or add more data."
    )
generator = torch.Generator().manual_seed(SEED) if SEED is not None else None
indices = torch.randperm(n_total, generator=generator).tolist()
train_set = torch.utils.data.Subset(train_dataset, indices[:train_size])
val_set = torch.utils.data.Subset(val_dataset, indices[train_size:])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train: {train_size} samples | Val: {val_size} samples")

Device: cuda
Train: 160 samples | Val: 40 samples


## Model

`DefaultCNN` is a small convolutional neural network with a few conv+pool layers followed by fully-connected layers that output class logits. It's intentionally simple — a baseline you can beat.

We use:
- **CrossEntropyLoss** — the standard loss for multi-class classification. It combines a softmax and a log-likelihood loss in one numerically stable operation.
- **Adam optimizer** — an adaptive gradient descent method that adjusts the learning rate per parameter. A sensible default for most classification tasks.

In [8]:
model = DefaultCNN(num_classes_=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

DefaultCNN(
  (conv1): Conv2d(3, 12, kernel_size=(2, 2), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(12, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (drop): Dropout2d(p=0.2, inplace=False)
  (fc): Linear(in_features=393216, out_features=4, bias=True)
)


## Training Loop

Each epoch is one full pass through the training set. Inside each epoch:

1. **`model.train()`** — switches the model to training mode. This matters because some layers (like Dropout and BatchNorm) behave differently during training vs. inference. Always call this before the training loop.
2. **Forward pass** — run a batch of images through the model to get output logits.
3. **Loss** — compare logits to true labels with CrossEntropyLoss.
4. **Backward pass** — call `loss.backward()` to compute gradients.
5. **Optimizer step** — call `optimizer.step()` to update weights. Always `zero_grad()` first, or gradients accumulate across batches.

We track both loss and accuracy per epoch so you can see whether the model is learning.

In [9]:
history: dict[str, list[float]] = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
}

for epoch in range(NUM_EPOCHS):
    # ── Training pass ──────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    correct = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()

    train_loss = running_loss / train_size
    train_acc = correct / train_size

    # ── Validation pass ────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    val_correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item() * images.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()

    val_loss /= val_size
    val_acc = val_correct / val_size

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch + 1:>2}/{NUM_EPOCHS} | "
        f"train loss: {train_loss:.4f}  acc: {train_acc:.3f} | "
        f"val loss: {val_loss:.4f}  acc: {val_acc:.3f}"
    )

Epoch  1/10 | train loss: 9.2348  acc: 0.294 | val loss: 1.3221  acc: 0.325
Epoch  2/10 | train loss: 1.2857  acc: 0.469 | val loss: 1.2868  acc: 0.375
Epoch  3/10 | train loss: 1.1561  acc: 0.531 | val loss: 1.1917  acc: 0.450
Epoch  4/10 | train loss: 0.9124  acc: 0.606 | val loss: 1.1122  acc: 0.375
Epoch  5/10 | train loss: 0.5150  acc: 0.850 | val loss: 1.1794  acc: 0.475
Epoch  6/10 | train loss: 0.3411  acc: 0.900 | val loss: 1.1773  acc: 0.425
Epoch  7/10 | train loss: 0.1684  acc: 0.969 | val loss: 1.2673  acc: 0.575
Epoch  8/10 | train loss: 0.1898  acc: 0.938 | val loss: 1.1262  acc: 0.550
Epoch  9/10 | train loss: 0.1254  acc: 0.981 | val loss: 1.1893  acc: 0.575
Epoch 10/10 | train loss: 0.0309  acc: 0.994 | val loss: 1.3215  acc: 0.425


## Save Model

We save `model.state_dict()` — the dictionary of trained weight tensors — not the entire model object. This is the standard PyTorch pattern: it's portable (doesn't depend on the class definition being importable at load time) and compact.

The saved file at `SAVE_PATH` is what the game loads when you run in CNN-assist mode. After training, restart the game and it will use your newly trained weights.

In [10]:
os.makedirs(os.path.dirname(SAVE_PATH) or ".", exist_ok=True)
torch.save(model.state_dict(), SAVE_PATH)
print(f"\nWeights saved to {SAVE_PATH}")


Weights saved to models\weights\baselineCNN.pth


## Training Plots

Two plots are saved to `CNN_LOG_DIR`:

- **`training_curves.png`** — train/val loss and accuracy side by side across epochs. If val accuracy starts dropping while train accuracy keeps rising, the model is overfitting.
- **`confusion_matrix.png`** — per-class prediction breakdown on the validation set. Shows which classes the model confuses most often.

### What you will see in the baseline run

With augmentation enabled, the baseline should overfit less than before — train accuracy will still climb faster than val accuracy, but the gap should be smaller and val accuracy should peak higher than the ~50% seen without augmentation.

**Why does overfitting still happen?** The dataset is small (~160 training images across 4 classes). Augmentation expands the effective variety the model sees each epoch, but it does not add new ground-truth examples. The network still has more parameters than the dataset has unique images, so some memorization is expected.

**This is still the floor you are meant to beat.** The module docstring lists further improvements: regularization (dropout, weight decay) and early stopping. Regularization directly penalizes memorization; early stopping saves weights at peak val accuracy before the model starts overfitting. Either should push val accuracy meaningfully higher.

In [11]:
def _save_cnn_plots(
    history: dict,
    model: DefaultCNN,
    val_loader: DataLoader,
    device: torch.device,
    class_names: list[str],
    log_dir: str,
) -> None:
    """Save training curve and confusion matrix plots to log_dir."""
    os.makedirs(log_dir, exist_ok=True)
    epochs_range = range(1, len(history["train_loss"]) + 1)

    # ── Training curves ───────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs_range, history["train_loss"], marker="o", label="Train")
    ax1.plot(epochs_range, history["val_loss"], marker="o", label="Val")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_range, history["train_acc"], marker="o", label="Train")
    ax2.plot(epochs_range, history["val_acc"], marker="o", label="Val")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy")
    ax2.legend()
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)

    fig.suptitle("CNN Training Curves", fontsize=14)
    plt.tight_layout()
    curves_path = os.path.join(log_dir, "training_curves.png")
    fig.savefig(curves_path, dpi=150)
    plt.close(fig)
    print(f"Training curves saved to {curves_path}")

    # ── Confusion matrix ──────────────────────────────────────────────────
    model.eval()
    all_preds: list[int] = []
    all_labels: list[int] = []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            all_preds.extend(outputs.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.tolist())

    n = len(class_names)
    cm = np.zeros((n, n), dtype=int)
    for true, pred in zip(all_labels, all_preds):
        cm[true][pred] += 1

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion Matrix (Validation Set)")

    thresh = cm.max() / 2.0
    for i in range(n):
        for j in range(n):
            ax.text(
                j, i, str(cm[i, j]),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=12,
            )

    plt.tight_layout()
    cm_path = os.path.join(log_dir, "confusion_matrix.png")
    fig.savefig(cm_path, dpi=150)
    plt.close(fig)
    print(f"Confusion matrix saved to {cm_path}")

In [12]:
class_names = [s.value for s in State]
_save_cnn_plots(history, model, val_loader, device, class_names, CNN_LOG_DIR)

Training curves saved to training/CNN-logs\training_curves.png
Confusion matrix saved to training/CNN-logs\confusion_matrix.png
